## Homework\n\nIn this homework, we'll deploy the Straight vs Curly Hair Type model we trained in the\n[previous homework](../08-deep-learning/homework.md).\n\nDownload the model files from here: \n\n* https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle/hair_classifier_v1.onnx.data\n* https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle/hair_classifier_v1.onnx\n\nWith wget:\n\n```bash\nPREFIX=\"https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle\"\nDATA_URL=\"${PREFIX}/hair_classifier_v1.onnx.data\"\nMODEL_URL=\"${PREFIX}/hair_classifier_v1.onnx\"\nwget ${DATA_URL}\nwget ${MODEL_URL}\n```

## Question 1\n\nTo be able to use this model, we need to know the name of the input and output nodes. \n\nWhat's the name of the output:\n\n* `output`\n* `sigmoid`\n* `softmax`\n* `prediction`

In [ ]:
import onnxruntime as ort\n\nmodel_path = 'hair_classifier_v1.onnx'\nsession = ort.InferenceSession(model_path)\ninput_name = session.get_inputs()[0].name\noutput_name = session.get_outputs()[0].name\n\nprint(f\"Input name: {input_name}\")\nprint(f\"Output name: {output_name}\")

**Answer:** The name of the output is `output`.

## Preparing the image\n\nYou'll need some code for downloading and resizing images. You can use \nthis code:\n\n```python\nfrom io import BytesIO\nfrom urllib import request\n\nfrom PIL import Image\n\ndef download_image(url):\n    with request.urlopen(url) as resp:\n        buffer = resp.read()\n    stream = BytesIO(buffer)\n    img = Image.open(stream)\n    return img\n\n\ndef prepare_image(img, target_size):\n    if img.mode != 'RGB':\n        img = img.convert('RGB')\n    img = img.resize(target_size, Image.NEAREST)\n    return img\n```\n\nFor that, you'll need to have `pillow` installed:\n\n```bash\npip install pillow\n```\n\n## Question 2: Target size\n\nLet's download and resize this image: \n\nhttps://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg\n\nBased on the previous homework, what should be the target size for the image?\n\n* 64x64\n* 128x128\n* 200x200\n* 256x256

In [ ]:
from io import BytesIO\nfrom urllib import request\nfrom PIL import Image\n\ndef download_image(url):\n    with request.urlopen(url) as resp:\n        buffer = resp.read()\n    stream = BytesIO(buffer)\n    img = Image.open(stream)\n    return img\n\n\ndef prepare_image(img, target_size):\n    if img.mode != 'RGB':\n        img = img.convert('RGB')\n    img = img.resize(target_size, Image.NEAREST)\n    return img\n\n# From homework 8, the target size is 200x200\ntarget_size = (200, 200)\nprint(f\"The target size is {target_size}\")

**Answer:** The target size is `200x200`.

## Question 3\n\nNow we need to turn the image into numpy array and pre-process it. \n\n> Tip: Check the previous homework. What was the pre-processing \n> we did there?\n\nAfter the pre-processing, what's the value in the first pixel, the R channel?\n\n* -10.73\n* -1.073\n* 1.073\n* 10.73

In [ ]:
import numpy as np\n\ndef preprocess_image(img):\n    x = np.array(img, dtype='float32')\n    # Scale to [0, 1]\n    x = x / 255.0\n    # Normalize with ImageNet mean and std\n    mean = np.array([0.485, 0.456, 0.406], dtype='float32')\n    std = np.array([0.229, 0.224, 0.225], dtype='float32')\n    x = (x - mean) / std\n    # Change from (H, W, C) to (C, H, W)\n    x = x.transpose(2, 0, 1)\n    return x\n\nimage_url = \"https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg\"\nimg = download_image(image_url)\nimg_prepared = prepare_image(img, target_size)\n\npreprocessed_image = preprocess_image(img_prepared)\n\n# For Question 3: value in the first pixel, the R channel\nfirst_pixel_r = preprocessed_image[0, 0, 0]\nprint(f\"The value of the R channel of the first pixel (after preprocessing) is: {first_pixel_r}\")

**Answer:** The value is approximately `-1.073`.

## Question 4\n\nNow let's apply this model to this image. What's the output of the model?\n\n* 0.09\n* 0.49\n* 0.69\n* 0.89

In [ ]:
import math\n\ndef sigmoid(x):\n  return 1 / (1 + math.exp(-x))\n\ninput_tensor = np.expand_dims(preprocessed_image, axis=0)\n\n# Run inference\nresult = session.run([output_name], {input_name: input_tensor})[0]\nprobability = sigmoid(result[0][0])\n\nprint(f\"Model output (logit): {result[0][0]}\")\nprint(f\"Model output (probability): {probability}\")

**Answer:** The output is approximately `0.89`.

## Question 5\n\nDownload the base image `agrigorev/model-2025-hairstyle:v1`. You can do it with [`docker pull`](https://docs.docker.com/engine/reference/commandline/pull/).\n\nSo what's the size of this base image?\n\n* 88 Mb\n* 208 Mb\n* 608 Mb\n* 1208 Mb\n\nYou can get this information when running `docker images` - it'll be in the \"SIZE\" column.

In [ ]:
!docker pull agrigorev/model-2025-hairstyle:v1

In [ ]:
!docker images agrigorev/model-2025-hairstyle

**Answer:** The size is `608 Mb`

## Question 6\n\nNow let's extend this docker image, install all the required libraries\nand add the code for lambda.\n\nYou don't need to include the model in the image. It's already included. \nThe name of the file with the model is `hair_classifier_empty.onnx` and it's \nin the current workdir in the image (see the Dockerfile above for the \nreference). \nThe provided model requires the same preprocessing for images regarding target size and rescaling the value range than used in homework 8.\n\nNow run the container locally.\n\nScore this image: https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg\n\nWhat's the output from the model?\n\n* -1.0\n* -0.10\n* 0.10\n* 1.0

Here's the `Dockerfile`:

In [ ]:
!cat Dockerfile

And the `lambda_function.py`:

In [ ]:
!cat lambda_function.py

Let's build the image.

In [ ]:
!docker build -t homework-image .

And now let's run it and get the prediction.

In [ ]:
!docker run -it --rm homework-image '{\"url\": \"https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg\"}'

**Answer:** The output from the model is `-0.10`.